In [ ]:
import pandas as pd, numpy as np, os

In [11]:
# load datasets
enc = pd.read_csv('encounters_with_policy.csv')
rep = pd.read_csv('repatriations_with_policy.csv')

# aggregate encounters monthly totals
enc_m = enc.groupby(['date','cyear','fiscal_year','month','administration','mpp_flag','title42_flag'], dropna = False)['quantity'].sum().reset_index()
enc_m = enc_m.rename(columns = {'quantity':'monthly_encounter_counts'})

# aggregate repatriations monthly totals and pivot key types
rep_tot = rep.groupby(['date','cyear','fiscal_year','month','administration','mpp_flag','title42_flag'], dropna=False)['quantity'].sum().reset_index()
rep_tot = rep_tot.rename(columns = {'quantity':'monthly_repatriation_counts'})

rep_piv = rep.pivot_table(index = ['date','cyear','fiscal_year','month','administration','mpp_flag','title42_flag'],
                          columns = 'repatriation_type', values = 'quantity', aggfunc = 'sum').reset_index()

# flatten columns
rep_piv.columns.name = None

# merge
master = enc_m.merge(rep_tot, on = ['date','cyear','fiscal_year','month','administration','mpp_flag','title42_flag'], how = 'outer')
master = master.merge(rep_piv, on  = ['date','cyear','fiscal_year','month','administration','mpp_flag','title42_flag'], how = 'left')

# rename common fields if exist
rename_map = {'Administrative Returns':'administrative_returns',
              'Enforcement Returns':'enforcement_returns',
              'Removals':'removals',
              'Title 42 Expulsions':'title42_expulsions'
              }
master = master.rename(columns = rename_map)

# ratio
master['repatriations_per_encounter'] = master['monthly_repatriation_counts'] / master['monthly_encounter_counts']

out = 'final_master_policy_dataset.csv'
master.to_csv(out,index = False)
print(out)
print(master.head())

combined_encounters_monthly_region_family.csv
        date  cyear  fiscal_year  month                       region  \
0 2003-10-01   2003         2004     10  Air Ports of Entry/Interior   
1 2003-10-01   2003         2004     10               Coastal Border   
2 2003-10-01   2003         2004     10         Northern Land Border   
3 2003-10-01   2003         2004     10        Southwest Land Border   
4 2003-11-01   2003         2004     11  Air Ports of Entry/Interior   

   quantity_x  family_status  quantity_y agency  
0        3000  Single Adults       10840    OFO  
1          70  Single Adults       10840    OFO  
2        3920  Single Adults       10840    OFO  
3        3860  Single Adults       10840    OFO  
4        2920  Single Adults       11350    OFO  


In [13]:
# load policy
policy = pd.read_csv('policy_markers_2015_2025.csv')
policy['start_date']=pd.to_datetime(policy['start_date'])
policy['end_date']=pd.to_datetime(policy['end_date'])

# build monthly flags table
dates = pd.date_range('2015-01-01','2025-12-01',freq = 'MS')
pm = pd.DataFrame({'date':dates})
pm['administration'] = ''

for _,r in policy.iterrows():
    mask = (pm['date'] >= r['start_date']) & (pm['date'] <=r ['end_date'])
    if r['category'] == 'administration':
        pm.loc[mask,'administration'] = r['admin']
    pid = str(r['policy_id']).lower()
    
    if 'mpp' in pid or 'remain' in str(r['policy_name']).lower():
        pm.loc[mask,'mpp_flag'] = 1
        
    if 'title42' in pid.replace(' ','') or 'title 42' in str(r['policy_name']).lower():
        pm.loc[mask,'title42_flag'] = 1
        
pm['mpp_flag'] = pm.get('mpp_flag',0).fillna(0).astype(int)
pm['title42_flag'] = pm.get('title42_flag',0).fillna(0).astype(int)

# encounters rebuild
def make_enc(path,agency):
    
    reg = pd.read_excel(path,sheet_name = 'Monthly Region')
    reg.columns = [c.replace('\n',' ').strip() for c in reg.columns]
    reg = reg.rename(columns = {'Fiscal Year':'fiscal_year','Quantity':'quantity'})
    
    if 'fiscal_year' not in reg.columns:
        for c in reg.columns:
            if 'Fiscal' in c: reg = reg.rename(columns = {c:'fiscal_year'})
                
    m = reg['Month'].astype(str)
    reg['month_num_fy'] = m.str.extract(r'(\d+)').astype(int)
    mapfy = {1:10,2:11,3:12,4:1,5:2,6:3,7:4,8:5,9:6,10:7,11:8,12:9}
    reg['month'] = reg['month_num_fy'].map(mapfy)
    reg['cyear'] = np.where(reg['month'] >= 10, reg['fiscal_year'] - 1, reg['fiscal_year'])
    reg['date'] = pd.to_datetime(dict(year = reg['cyear'],month = reg['month'],day = 1))
    reg['agency'] = agency
    
    return reg[['date','cyear','fiscal_year','month','Region','quantity','agency']].rename(columns = {'Region':'region'})

enc = pd.concat([make_enc('KHSM Encounters (OFO) fy25m11.xlsx','OFO'),
               make_enc('KHSM Encounters (USBP) fy25m11.xlsx','USBP')
               ],ignore_index = True)

enc = enc.merge(pm,on = 'date',how = 'left')
enc_out = 'encounters_with_policy.csv'
enc.to_csv(enc_out,index = False)

# repat rebuild
rep = pd.read_excel('KHSM Repatriations fy25m11.xlsx',sheet_name = 'Monthly Repatriation Type')
rep.columns = [c.replace('\n',' ').strip() for c in rep.columns]
rep = rep.rename(columns = {'Fiscal Year':'fiscal_year','Repatriation Type':'repatriation_type','Quantity':'quantity'})

if 'fiscal_year' not in rep.columns:
    for c in rep.columns:
        if 'Fiscal' in c: rep = rep.rename(columns = {c:'fiscal_year'})
            
m = rep['Month'].astype(str)
rep['month_num_fy'] = m.str.extract(r'(\d+)').astype(int)
mapfy = {1:10,2:11,3:12,4:1,5:2,6:3,7:4,8:5,9:6,10:7,11:8,12:9}
rep['month'] = rep['month_num_fy'].map(mapfy)
rep['cyear'] = np.where(rep['month'] >= 10, rep['fiscal_year'] - 1, rep['fiscal_year'])
rep['date'] = pd.to_datetime(dict(year = rep['cyear'],month = rep['month'],day = 1))
rep = rep[['date','cyear','fiscal_year','month','repatriation_type','quantity']]
rep = rep.merge(pm,on = 'date',how = 'left')
rep_out = 'repatriations_datasource.csv'
rep.to_csv(rep_out,index = False)

print(enc_out)
print(rep_out)

encounters_with_policy.csv
repatriations_datasource.csv
